# 2 Robot Tasks (REC and SEN)
# 2 C++ Threads (receive_thread, send_thread)
    move +- 10 deg (10 deg/s)

    TRY: SCHED_RR on same CPU

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from urdfpy import URDF
from urdfpy import matrix_to_rpy
import plotly.express as px

Load data

In [6]:
def get_arrays():
    act_df = pd.read_csv("act.csv")
    com_df = pd.read_csv("com.csv")

    timestamps = act_df["timestamp"].values
    timestamps = np.array([(t - timestamps[0]) / 1000 for t in timestamps])

    # 1. Sicherstellen, dass timestamp ein datetime ist
    com_df['timestamp'] = pd.to_datetime(com_df['timestamp'])
    act_df['timestamp'] = pd.to_datetime(act_df['timestamp'])

    # 2. Index setzen für Zeitreiheninterpolation
    com_df_indexed = com_df.set_index('timestamp')
    act_timestamps = act_df['timestamp']

    # 3. Reindexieren mit kombinierten Timestamps (Index.union ist erlaubt!)
    combined_index = com_df_indexed.index.union(act_timestamps)
    com_df_reindexed = com_df_indexed.reindex(combined_index).sort_index()

    # 4. Interpolation entlang Zeitachse
    com_df_interpolated = com_df_reindexed.interpolate(method='time')

    # 5. Nur die Zeilen mit act_df timestamps behalten
    com_df_new = com_df_interpolated.loc[act_timestamps].reset_index()

    # Optional: nur j1 bis j6 und timestamp
    # com_df_new = com_df_new[['timestamp', 'j1', 'j2', 'j3', 'j4', 'j5', 'j6']]

    act_j = act_df.drop(columns=["timestamp"]).values
    com_j = com_df_new.drop(columns=["timestamp"]).values

    mask = ~np.isnan(com_j).any(axis=1)
    act_j = act_j[mask, :]
    com_j = com_j[mask, :]
    timestamps = timestamps[mask]

    j_err = act_j - com_j
    print(j_err[(int(j_err.argmax() / 6))])
    print(j_err.max())

    return act_j, com_j, timestamps

calculate errors in position and orientation

In [7]:
def get_errors(act_j, com_j):
    robot = URDF.load("../../kuka_kr60_moveit_config/urdf/kr60.urdf")

    orientation_act = []
    pos_act = []
    orientation_com = []
    pos_com = []

    for i in range(act_j.shape[0]):
        cfg_act = {
            "joint_a1": np.deg2rad(act_j[i][0]),
            "joint_a2": np.deg2rad(act_j[i][1]),
            "joint_a3": np.deg2rad(act_j[i][2]),
            "joint_a4": np.deg2rad(act_j[i][3]),
            "joint_a5": np.deg2rad(act_j[i][4]),
            "joint_a6": np.deg2rad(act_j[i][5])
        }

        cfg_com = {
            "joint_a1": np.deg2rad(com_j[i][0]),
            "joint_a2": np.deg2rad(com_j[i][1]),
            "joint_a3": np.deg2rad(com_j[i][2]),
            "joint_a4": np.deg2rad(com_j[i][3]),
            "joint_a5": np.deg2rad(com_j[i][4]),
            "joint_a6": np.deg2rad(com_j[i][5])
        }

        T_act = robot.link_fk(
            cfg=cfg_act,
            link="link_6"
        )

        T_com = robot.link_fk(
            cfg=cfg_com,
            link="link_6"
        )

        orientation_act.append(T_act[:3, :3])
        pos_act.append(T_act[:3, -1])

        orientation_com.append(T_com[:3, :3])
        pos_com.append(T_com[:3, -1])

    pos_err = []
    for pact, pcom in zip(pos_act, pos_com):
        err = 1.0e6 * (pact - pcom)
        pos_err.append(np.sqrt(err[0]**2 + err[1]**2 + err[2]**2))
    
    rpy_act = []
    rpy_com = []
    for oact, ocom in zip(orientation_act, orientation_com):
        rpy = matrix_to_rpy(oact)
        rpy = np.array([np.rad2deg(r) for r in rpy])
        rpy_act.append(rpy)

        rpy = matrix_to_rpy(ocom)
        rpy = np.array([np.rad2deg(r) for r in rpy])
        rpy_com.append(rpy)
    
    rpy_act = np.array(rpy_act)
    rpy_com = np.array(rpy_com)
    
    return pos_err, rpy_act, rpy_com

Plot distance error

In [8]:
act_j, com_j, timestamps = get_arrays()
pos_err, rpy_act, rpy_com = get_errors(act_j, com_j)

# 1) DataFrame anlegen
df = pd.DataFrame({
    'timestamp': timestamps,
    'error_um': pos_err
})

# 2) Interaktiven Linienplot
fig = px.line(
    df,
    x='timestamp',
    y='error_um',
    title="Position Error über Zeit",
    labels={
        'timestamp': 'Timestamp',
        'error_um': 'Position Error [µm]'
    }
)

# 3) Achsen-Datumsformat nach Wunsch
fig.update_xaxes(
    tickformat="%H:%M:%S\n%d-%m-%Y",
    # rangeslider_visible=True     # kleiner Slider unten für grobes Zoomen
)

fig.show()

[4.92195704 4.91393704 4.91839704 4.92385704 4.92402704 4.92721704]
4.927217037037037


orientation over time

In [9]:
# 1) DataFrame anlegen
df = pd.DataFrame({
    'timestamp': timestamps,
    'act_x': rpy_act[:, 0],
    'act_y': rpy_act[:, 1],
    'act_z': rpy_act[:, 2],
    'com_x': rpy_com[:, 0],
    'com_y': rpy_com[:, 1],
    'com_z': rpy_com[:, 2],
})

# 1) In Long-Form umwandeln
df_long = df.melt(
    id_vars='timestamp',
    value_vars=df.columns[1:],
    var_name='Channel',
    value_name='Orientation [deg]'
)

# 2) Plot bauen
fig = px.line(
    df_long,
    x='timestamp',
    y='Orientation [deg]',
    color='Channel',
    title="Orientation über Zeit",
    labels={'timestamp': 'Timestamp'}
)

# 3) Achsen anpassen
fig.update_xaxes(
    tickformat="%H:%M:%S\n%d-%m-%Y",
)

fig.show()

Joint angles over time

In [10]:
# 1) DataFrame anlegen
df = pd.DataFrame({
    'timestamp': timestamps,
    'act_j1': act_j[:, 0],
    'act_j2': act_j[:, 1],
    'act_j3': act_j[:, 2],
    'act_j4': act_j[:, 3],
    'act_j5': act_j[:, 4],
    'act_j6': act_j[:, 5],
    'com_j1': com_j[:, 0],
    'com_j2': com_j[:, 1],
    'com_j3': com_j[:, 2],
    'com_j4': com_j[:, 3],
    'com_j5': com_j[:, 4],
    'com_j6': com_j[:, 5]
})

# 1) In Long-Form umwandeln
df_long = df.melt(
    id_vars='timestamp',
    value_vars=df.columns[1:],
    var_name='Channel',
    value_name='joints [deg]'
)

# 2) Plot bauen
fig = px.line(
    df_long,
    x='timestamp',
    y='joints [deg]',
    color='Channel',
    title="Joint angles over time",
    labels={'timestamp': 'Timestamp'}
)

# 3) Achsen anpassen
fig.update_xaxes(
    tickformat="%H:%M:%S\n%d-%m-%Y",
)

fig.show()